In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from datasets import load_from_disk
# from peft import PeftModel
from utils import *

os.environ['CUDA_VISIBLE_DEVICES'] = "5,6"

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    'meta-llama/Llama-2-7b-chat-hf',
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.float16,   
)
base_model.model_parellal = True

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(base_model, 'FinGPT/fingpt-forecaster_dow30_llama2-7b_lora')
model = model.eval()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-2-7b-chat-hf')
tokenizer.padding_side = "right"
tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
test_dataset = load_from_disk('./data/fingpt-forecaster-crypto-20230131-20231231-1-4-08/')['test']

In [ ]:
def test_demo(model, tokenizer, prompt):

    inputs = tokenizer(
        prompt, return_tensors='pt',
        padding=False, max_length=4096,
        truncation=True
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
        
    res = model.generate(
        **inputs, max_new_tokens=512, do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True
    )
    output = tokenizer.decode(res[0], skip_special_tokens=True)
    return output    
    # return res

In [ ]:
answers, gts = [], []

for i in range(len(test_dataset)):
    prompt = test_dataset[i]['prompt']
    output = test_demo(model, tokenizer, prompt)
    answer = re.sub(r'.*\[/INST\]\s*', '', output, flags=re.DOTALL)
    gt = test_dataset[i]['answer']
    print('\n------- Prompt ------\n')
    print(prompt)
    print('\n------- LLaMA2 Finetuned ------\n')
    print(answer)
    print('\n------- GPT4 Groundtruth ------\n')
    print(gt)
    print('\n===============\n')
    answers.append(answer)
    gts.append(gt)

In [ ]:
gts[0]

In [ ]:
answers[0]

In [ ]:
calc_metrics(answers, gts)

In [ ]:
print(answers[0])